# Practice: drag and the along-track error horizon

A sandbox for `orbit_correction.truth_prop`'s drag. Drag lowers an orbit; a lower orbit
moves faster; so an error in drag shows up as an **along-track** error that grows roughly
with the **square** of the time since the start. Solar activity sets the density, so a
wrong solar-activity input is one of the main ways a propagator drifts.

This notebook propagates one LEO object with the project's truth configuration, then
repeats it with the F10.7 solar flux scaled up or down in a *copy* of the space-weather
file, and measures the along-track difference at increasing horizons.

Inputs: the committed space-weather fixture (`tests/fixtures/sw_celestrak_2024_extract.txt`,
observed 2023-10-01 to 2024-12-31). The scaled copies go to a temporary directory; nothing
here writes to `src/` or `data/`. Needs `uv sync --group datagen`.

In [ ]:
import dataclasses
import tempfile
from pathlib import Path

import numpy as np

from orbit_correction import truth_prop as T
from orbit_correction.frames import utc_to_tt_seconds
from orbit_correction.spaceweather import parse_spaceweather
from orbit_correction.types import Trajectory

REPO = T.REPO  # the repository root (the package is installed editable)

DAY = 86400.0
# Truth starts from one state here, so the epoch-state rule is named explicitly: the
# configured fitted start (2026-11-25) needs SGP4 samples before the epoch.
cfg = dataclasses.replace(T.load_truth_config(), start_rule=T.EPOCH_START_RULE)
ctx = T._context(cfg)  # starts this kernel's JVM (once; restart the kernel to change JVM options)
MU = ctx.mu            # m^3/s^2, EIGEN-6S
print("zonal degree in config:", cfg.zonal_degree, "| mu:", MU)

SW_FIXTURE = REPO / "tests" / "fixtures" / "sw_celestrak_2024_extract.txt"
TMP = Path(tempfile.mkdtemp(prefix="drag_horizon_"))
SW = parse_spaceweather(SW_FIXTURE)

## A space-weather copy with scaled solar flux

`scaled_copy(k)` multiplies every F10.7 value (daily and 81-day means, observed and
adjusted) by `k` and writes an Orekit copy with the project's writer. `k = 1` is the real
data. Geomagnetic indices are left alone.

In [ ]:
START_UTC, END_UTC = "2024-03-01", "2024-03-08"

def scaled_copy(k):
    names = ("f107_obs", "f107_obs_ctr81", "f107_obs_lst81", "f107_adj", "f107_adj_ctr81",
             "f107_adj_lst81")
    scalars = dict(SW.scalars)
    for n in names:
        scalars[n] = SW.scalars[n] * k
    path = TMP / f"sw_k{k:.2f}.txt"
    # truth_prop writes the Orekit copy and records the range it was checked for;
    # propagate_truth refuses windows outside it
    return T.write_space_weather_copy(
        dataclasses.replace(SW, scalars=scalars), path,
        utc_to_tt_seconds(START_UTC + "T00:00:00"), utc_to_tt_seconds(END_UTC + "T12:00:00"))

print(scaled_copy(1.0).path)

## The object

A 400 km circular orbit at 51.6° with A/m from a typical B* (2 × 10⁻⁴ /ER). Change these.

In [ ]:
ALT_KM, INC_DEG, BSTAR = 400.0, 51.6, 2e-4
t0 = utc_to_tt_seconds(START_UTC + "T00:07:00")

r = 6378.137 + ALT_KM
v = np.sqrt(MU / 1e9 / r)
i = np.radians(INC_DEG)
init = Trajectory(1, np.array([t0]), np.array([[r, 0, 0]]),
                  np.array([[0, v * np.cos(i), v * np.sin(i)]]), "practice")
grid = T.output_grid(t0, 7 * DAY, 60.0)
print("A/m from B*:", T.area_to_mass_from_bstar(BSTAR, cfg), "m^2/kg")

def along_track_m(ref, other):
    """Curvilinear along-track separation (other ahead of ref is positive), metres."""
    h = np.cross(ref.r_gcrf_km, ref.v_gcrf_km_s)
    n = h / np.linalg.norm(h, axis=1)[:, None]
    ang = np.arctan2(np.einsum("ij,ij->i", np.cross(ref.r_gcrf_km, other.r_gcrf_km), n),
                     np.einsum("ij,ij->i", ref.r_gcrf_km, other.r_gcrf_km))
    return ang * np.linalg.norm(ref.r_gcrf_km, axis=1) * 1000.0

base = T.propagate_truth(init, BSTAR, grid, cfg, scaled_copy(1.0)).trajectory

## Scale the solar flux and watch the error grow

Rows are horizons, columns are flux scale factors. Sign: positive means the run with
scaled flux is *ahead* of the real-flux run (more drag, lower, faster).

In [ ]:
HORIZONS_H = (1, 6, 24, 72, 168)
idx = [int(h * 3600 / 60) for h in HORIZONS_H]
runs = {}
for k in (0.8, 0.9, 1.1, 1.2, 1.5):
    tr = T.propagate_truth(init, BSTAR, grid, cfg, scaled_copy(k)).trajectory
    runs[k] = along_track_m(base, tr)
no_drag = T.propagate_truth(init, None, grid, dataclasses.replace(cfg, atmosphere="none"), None)
runs["no drag"] = along_track_m(base, no_drag.trajectory)

print(f"{'horizon':>8} " + " ".join(f"{str(k):>10}" for k in runs))
for h, j in zip(HORIZONS_H, idx, strict=True):
    print(f"{h:6d} h " + " ".join(f"{runs[k][j]:10.1f}" for k in runs))

## Is it quadratic?

If the along-track error is ½·c·t², then error/t² is constant. Check it for the
k = 1.2 run (early horizons are noisy: the error is still small there).

In [ ]:
d = runs[1.2]
for h, j in zip(HORIZONS_H, idx, strict=True):
    print(f"{h:4d} h: along-track {d[j]:9.1f} m, / t^2 = {d[j] / (h * 3600.0) ** 2:.3e} m/s^2")

## Things to try

- Lower the altitude to 300 km or raise it to 800 km: the error changes by orders of
  magnitude (density falls roughly exponentially with altitude).
- Move the start to the May 2024 storm (`START_UTC = "2024-05-08"`) and scale only the
  geomagnetic indices (edit `scaled_copy` to change `SW.ap`).
- Change `cfg.msis_ap_switch` between -1 and 1 with `dataclasses.replace` and compare: how
  much does using the 3-hourly ap history matter in a storm?
- Double `BSTAR`: the along-track error scales with A/m too, which is why B* matters.